# 水果识别 CNN 训练
## Colab GPU 一键训练

> Runtime → Change runtime type → **T4 GPU**

### Step 0 — 首次使用只需一次
1. kaggle.com/settings → Create New Token → 下载 `kaggle.json`
2. 每次重连 Colab 后，**左侧 Files 面板** → 右键 Upload → 上传 `kaggle.json`
3. 然后从 Step 1 开始顺序执行

In [ ]:
# 1. 配置 Kaggle API
import os, shutil
os.makedirs('/root/.kaggle', exist_ok=True)
if os.path.exists('kaggle.json'):
    shutil.copy('kaggle.json', '/root/.kaggle/kaggle.json')
    os.chmod('/root/.kaggle/kaggle.json', 0o600)
    print('Kaggle API 就绪')
else:
    print('请先上传 kaggle.json 到 Colab Files 面板')
    print('kaggle.com/settings → Create New Token → 下载')

In [ ]:
# 2. GPU 检测
import torch
print(f"PyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else '未检测到'}")

In [ ]:
# 3. 安装依赖
!pip install -q torchvision scikit-learn matplotlib seaborn pillow kaggle
!apt-get install -y -q aria2 2>/dev/null
print('依赖就绪')

In [ ]:
# 4. 下载数据集 (kagglehub + symlink)
import os, shutil, subprocess
from pathlib import Path

os.makedirs('data/raw', exist_ok=True)
dest = Path('data/raw/fruits-360')

try:
    import kagglehub
    cached = kagglehub.dataset_download('moltean/fruits')
    print(f"kagglehub: {cached}")
    if not dest.exists():
        dest.symlink_to(Path(cached).resolve())
    total = sum(1 for _ in dest.rglob('*.jpg'))
    print(f"数据集: {total} 张")
except Exception as e:
    print(f"kagglehub 失败: {e}")
    print("尝试 aria2c...")
    r = subprocess.run(['kaggle','datasets','download','-d','moltean/fruits','-w'],
                       capture_output=True, text=True)
    url = r.stdout.strip().split('\n')[-1]
    if url.startswith('http'):
        subprocess.run(['aria2c','-x','16','-s','16','-o','fruits-360.zip',url], check=True)
        import zipfile
        dest.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile('fruits-360.zip','r') as zf:
            zf.extractall(dest)
        os.remove('fruits-360.zip')
        print(f"数据集: {sum(1 for _ in dest.rglob('*.jpg'))} 张")
    else:
        print("下载失败。请手动上传 data/processed/ 到 Google Drive。")

In [ ]:
# 5. 挂载 Google Drive（仅保存模型用）
from google.colab import drive
drive.mount('/content/drive')
import os
drive_models = '/content/drive/MyDrive/fruit-model'
os.makedirs(drive_models, exist_ok=True)
print(f"模型备份路径: {drive_models}")

In [ ]:
# 6. 创建 src 目录
import os
from pathlib import Path
for d in ['src/utils','src/data','src/models','src/training','src/evaluation']:
    os.makedirs(d, exist_ok=True)
Path('src/__init__.py').touch()

In [ ]:
%%writefile src/utils/config.py
import random, numpy as np, torch
from dataclasses import dataclass, field
from pathlib import Path
from typing import List, Tuple

@dataclass
class Config:
    project_root: Path = Path('.')
    data_raw_dir: Path = Path('data/raw')
    data_processed_dir: Path = Path('data/processed')
    models_dir: Path = Path('models')
    figures_dir: Path = Path('docs/report/figures')
    target_classes: List[str] = field(default_factory=lambda: [
        "Apple","Banana","Blueberry","Cherry","Grape",
        "Kiwi","Lemon","Mango","Orange","Peach",
        "Pear","Pineapple","Pomegranate","Strawberry","Watermelon",
    ])
    img_size: Tuple[int,int] = (128,128)
    train_split: float = 0.8
    val_split_from_train: float = 0.125
    random_seed: int = 42
    batch_size: int = 32
    num_epochs: int = 50
    num_classes: int = 15
    learning_rate: float = 0.001
    weight_decay: float = 1e-4
    dropout_rate: float = 0.5
    scheduler_factor: float = 0.1
    scheduler_patience: int = 5
    early_stopping_patience: int = 10
    early_stopping_delta: float = 0.0
    normalize_mean: Tuple[float,float,float] = (0.485,0.456,0.406)
    normalize_std: Tuple[float,float,float] = (0.229,0.224,0.225)
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
%%writefile src/utils/nutrition.py
from typing import Dict, List

CLASS_NAMES: List[str] = [
    "Apple","Banana","Blueberry","Cherry","Grape",
    "Kiwi","Lemon","Mango","Orange","Peach",
    "Pear","Pineapple","Pomegranate","Strawberry","Watermelon",
]
CLASS_NAMES_ZH: List[str] = [
    "苹果","香蕉","蓝莓","樱桃","葡萄",
    "猕猴桃","柠檬","芒果","橙子","桃子",
    "梨","菠萝","石榴","草莓","西瓜",
]
FRUIT360_TO_STANDARD: Dict[str,str] = {
    "Apple Red Delicious":"Apple","Apple Red Delicious 1":"Apple",
    "Apple Golden":"Apple","Apple Golden 1":"Apple","Apple Golden 2":"Apple","Apple Golden 3":"Apple",
    "Apple Granny Smith":"Apple","Apple Granny Smith 1":"Apple",
    "Apple Crimson Snow":"Apple","Apple Crimson Snow 1":"Apple",
    "Apple Red":"Apple","Apple Red 1":"Apple","Apple Red 2":"Apple","Apple Red 3":"Apple",
    "Apple Red Yellow 1":"Apple","Apple Red Yellow 2":"Apple",
    "Apple Yellow":"Apple","Apple Yellow 1":"Apple",
    "Apple Pink Lady":"Apple","Apple Pink Lady 1":"Apple",
    "Apple Braeburn":"Apple","Apple Braeburn 1":"Apple",
    "Banana":"Banana","Banana 1":"Banana","Banana 3":"Banana","Banana 4":"Banana",
    "Banana Red":"Banana","Banana Red 1":"Banana",
    "Banana Lady Finger":"Banana","Banana Lady Finger 1":"Banana",
    "Orange":"Orange","Orange 1":"Orange","Orange 2":"Orange","Orange 3":"Orange","orange 4":"Orange",
    "Grape":"Grape","Grape White":"Grape","Grape White 1":"Grape",
    "Grape White 2":"Grape","Grape White 3":"Grape","Grape White 4":"Grape",
    "Grape Pink":"Grape","Grape Pink 1":"Grape","Grape pink 2":"Grape",
    "Grape Blue":"Grape","Grape Blue 1":"Grape","Grape 1":"Grape",
    "Strawberry":"Strawberry","Strawberry 1":"Strawberry","Strawberry 2":"Strawberry",
    "Strawberry 3":"Strawberry","Strawberry Wedge":"Strawberry","Strawberry Wedge 1":"Strawberry",
    "Kiwi":"Kiwi","Kiwi 1":"Kiwi","Watermelon":"Watermelon","Watermelon 1":"Watermelon",
    "Peach":"Peach","Peach 1":"Peach","Peach 2":"Peach","Peach 3":"Peach",
    "Peach 4":"Peach","Peach 5":"Peach","Peach 6":"Peach",
    "Peach Flat":"Peach","Peach Flat 1":"Peach",
    "Pear":"Pear","Pear 1":"Pear","Pear 2":"Pear","Pear 3":"Pear",
    "Pear 5":"Pear","Pear 6":"Pear","Pear 7":"Pear","Pear 8":"Pear",
    "Pear 9":"Pear","Pear 10":"Pear","Pear 11":"Pear","Pear 12":"Pear",
    "Pear 13":"Pear","Pear 14":"Pear",
    "Pear Red":"Pear","Pear Red 1":"Pear",
    "Pear Kaiser":"Pear","Pear Kaiser 1":"Pear",
    "Pear Abate":"Pear","Pear Abate 1":"Pear",
    "Pear Forelle":"Pear","Pear Forelle 1":"Pear",
    "Pear Monster":"Pear","Pear Monster 1":"Pear",
    "Pear Williams":"Pear","Pear Williams 1":"Pear",
    "Pear Stone":"Pear","Pear Stone 1":"Pear",
    "Mango":"Mango","Mango 1":"Mango","Mango Red":"Mango","Mango Red 1":"Mango",
    "Pineapple":"Pineapple","Pineapple 1":"Pineapple",
    "Pineapple Mini":"Pineapple","Pineapple Mini 1":"Pineapple",
    "Cherry":"Cherry","Cherry 1":"Cherry","Cherry 2":"Cherry",
    "Cherry 3":"Cherry","Cherry 4":"Cherry","Cherry 5":"Cherry",
    "Cherry Rainier":"Cherry","Cherry Rainier 1":"Cherry",
    "Cherry Rainier 2":"Cherry","Cherry Rainier 3":"Cherry",
    "Cherry Wax":"Cherry","Cherry Wax 1":"Cherry","Cherry Wax 2":"Cherry",
    "Cherry Wax Red":"Cherry","Cherry Wax Red 1":"Cherry",
    "Cherry Wax Red 2":"Cherry","Cherry Wax Red 3":"Cherry",
    "Cherry Wax Black":"Cherry","Cherry Wax Black 1":"Cherry",
    "Cherry Wax Yellow":"Cherry","Cherry Wax Yellow 1":"Cherry",
    "Cherry Sour":"Cherry","Cherry Sour 1":"Cherry",
    "Lemon":"Lemon","Lemon 1":"Lemon","Lemon Meyer":"Lemon","Lemon Meyer 1":"Lemon",
    "Blueberry":"Blueberry","Blueberry 1":"Blueberry",
    "Pomegranate":"Pomegranate","Pomegranate 1":"Pomegranate",
}

In [ ]:
%%writefile src/data/make_dataset.py
import os, shutil
from pathlib import Path
from typing import Dict, List
from sklearn.model_selection import train_test_split
from src.utils.nutrition import CLASS_NAMES, FRUIT360_TO_STANDARD

def ensure_dir(path):
    path.mkdir(parents=True, exist_ok=True)
    return path

def _match_class(folder_name, target_names):
    exact = FRUIT360_TO_STANDARD.get(folder_name)
    if exact: return exact
    name = folder_name.lower()
    for t in target_names:
        if name == t.lower() or name.startswith(t.lower()+" "):
            return t
    return None

def collect_class_images(data_root, target_names):
    result = {}
    for folder in sorted(Path(data_root).iterdir()):
        if not folder.is_dir(): continue
        std = _match_class(folder.name, target_names)
        if std is None: continue
        imgs = sorted(p for p in folder.iterdir()
                      if p.suffix.lower() in ('.jpg','.jpeg','.png'))
        if imgs: result.setdefault(std,[]).extend(imgs)
    return result

def stratified_split(class_images, train_ratio=0.8, seed=42):
    train, test = {}, {}
    for cls, paths in class_images.items():
        t, te = train_test_split(paths, test_size=1-train_ratio, random_state=seed, shuffle=True)
        train[cls], test[cls] = t, te
    return train, test

def copy_images(split, dest_root, split_name):
    count = 0
    for cls, paths in split.items():
        d = ensure_dir(Path(dest_root) / split_name / cls)
        for src in paths:
            shutil.copy2(src, d/src.name); count += 1
    return count

def main():
    raw = Path('data/raw/fruits-360')
    processed = Path('data/processed')
    if (processed/'train').exists() and (processed/'test').exists():
        print('数据已处理，跳过')
        return
    train_dir = raw/'Training'
    test_dir = raw/'Test'
    if not train_dir.exists():
        raise FileNotFoundError(f'未找到: {train_dir}')
    train_imgs = collect_class_images(train_dir, CLASS_NAMES)
    test_raw = collect_class_images(test_dir, CLASS_NAMES) if test_dir.exists() else {}
    all_imgs = {}
    for c in sorted(set(list(train_imgs)+list(test_raw))):
        all_imgs[c] = train_imgs.get(c,[])+test_raw.get(c,[])
    n = sum(len(v) for v in all_imgs.values())
    print(f'{len(all_imgs)} 类, {n} 张')
    train_d, test_d = stratified_split(all_imgs, 0.8, 42)
    if processed.exists(): shutil.rmtree(processed)
    print(f'Train: {copy_images(train_d,processed,"train")} / Test: {copy_images(test_d,processed,"test")}')

if __name__ == '__main__':
    main()

In [ ]:
%%writefile src/data/fruit_dataset.py
from pathlib import Path
import torch
from torch.utils.data import DataLoader, Dataset, Subset
from PIL import Image
import torchvision.transforms as transforms
from sklearn.model_selection import train_test_split

class FruitDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        root = Path(root_dir)
        self.classes = sorted([d.name for d in root.iterdir() if d.is_dir()])
        self.class_to_idx = {c:i for i,c in enumerate(self.classes)}
        self.samples = []
        self.transform = transform
        for c in self.classes:
            for p in sorted((root/c).iterdir()):
                if p.suffix.lower() in ('.jpg','.jpeg','.png'):
                    self.samples.append((str(p), self.class_to_idx[c]))
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        p, l = self.samples[idx]
        img = Image.open(p).convert('RGB')
        return (self.transform(img), l) if self.transform else (img, l)

def get_train_transform(img_size=(128,128)):
    return transforms.Compose([
        transforms.Resize(img_size),
        transforms.RandomHorizontalFlip(0.5),
        transforms.RandomRotation(10),
        transforms.ColorJitter(0.2,0.2,0.2,0.05),
        transforms.ToTensor(),
        transforms.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225)),
    ])

def get_test_transform(img_size=(128,128)):
    return transforms.Compose([
        transforms.Resize(img_size),
        transforms.ToTensor(),
        transforms.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225)),
    ])

def get_dataloaders(data_dir, batch_size=32, img_size=(128,128), val_split=0.125, seed=42):
    data_dir = Path(data_dir)
    train_ds = FruitDataset(data_dir/'train', get_train_transform(img_size))
    labels = [l for _,l in train_ds.samples]
    tr, va = train_test_split(range(len(train_ds)), test_size=val_split,
                              stratify=labels, random_state=seed)
    val_ds = FruitDataset(data_dir/'train', get_test_transform(img_size))
    test_ds = FruitDataset(data_dir/'test', get_test_transform(img_size))
    return (
        DataLoader(Subset(train_ds,tr), batch_size, shuffle=True),
        DataLoader(Subset(val_ds,va), batch_size, shuffle=False),
        DataLoader(test_ds, batch_size, shuffle=False),
    )

In [ ]:
%%writefile src/models/cnn_model.py
import torch
import torch.nn as nn

class FruitCNN(nn.Module):
    def __init__(self, num_classes=15, dropout_rate=0.5):
        super().__init__()
        self.block1 = nn.Sequential(nn.Conv2d(3,32,3,1,1),nn.BatchNorm2d(32),nn.ReLU(True),nn.MaxPool2d(2))
        self.block2 = nn.Sequential(nn.Conv2d(32,64,3,1,1),nn.BatchNorm2d(64),nn.ReLU(True),nn.MaxPool2d(2))
        self.block3 = nn.Sequential(nn.Conv2d(64,128,3,1,1),nn.BatchNorm2d(128),nn.ReLU(True),nn.MaxPool2d(2))
        self.block4 = nn.Sequential(nn.Conv2d(128,256,3,1,1),nn.BatchNorm2d(256),nn.ReLU(True),nn.MaxPool2d(2))
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(dropout_rate)
        self.fc1 = nn.Linear(256,128)
        self.fc2 = nn.Linear(128,num_classes)
        self.relu = nn.ReLU(True)

    def forward(self,x):
        x = self.block4(self.block3(self.block2(self.block1(x))))
        x = torch.flatten(self.pool(x),1)
        x = self.relu(self.fc1(self.dropout(x)))
        return self.fc2(x)

In [ ]:
%%writefile src/training/trainer.py
import csv, math
from pathlib import Path
from typing import Dict, List, Tuple, Union
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader

class EarlyStopping:
    def __init__(self, patience=10, min_delta=0.0):
        self.patience, self.min_delta = patience, min_delta
        self.counter, self.best_loss, self.early_stop = 0, float('inf'), False
    def __call__(self, val_loss):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss, self.counter = val_loss, 0
        else:
            self.counter += 1
            if self.counter >= self.patience: self.early_stop = True
        return self.early_stop

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    loss_sum, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward(); optimizer.step()
        loss_sum += loss.item()*imgs.size(0)
        _, preds = torch.max(model(imgs), 1)
        total += labels.size(0); correct += (preds==labels).sum().item()
    return loss_sum/total, 100.*correct/total

@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        loss = criterion(model(imgs), labels)
        loss_sum += loss.item()*imgs.size(0)
        _, preds = torch.max(model(imgs), 1)
        total += labels.size(0); correct += (preds==labels).sum().item()
    return loss_sum/total, 100.*correct/total

def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler,
                config, num_epochs=50, checkpoint_dir='models', log_path='training_log.csv'):
    Path(checkpoint_dir).mkdir(parents=True, exist_ok=True)
    device = next(model.parameters()).device
    es = EarlyStopping(config.early_stopping_patience, config.early_stopping_delta)
    history = {'train_loss':[],'train_acc':[],'val_loss':[],'val_acc':[],'lr':[]}
    best_acc, best_ep = 0.0, 0
    with open(log_path,'w',newline='') as f:
        csv.writer(f).writerow(['epoch','train_loss','train_acc','val_loss','val_acc','lr'])
    for ep in range(1, num_epochs+1):
        tl, ta = train_epoch(model, train_loader, criterion, optimizer, device)
        vl, va = validate(model, val_loader, criterion, device)
        lr = optimizer.param_groups[0]['lr']
        for k,v in zip(['train_loss','train_acc','val_loss','val_acc','lr'],[tl,ta,vl,va,lr]):
            history[k].append(v)
        print(f'Epoch {ep:2d}/{num_epochs} | Train Loss: {tl:.4f} | Train Acc: {ta:.2f}% | Val Loss: {vl:.4f} | Val Acc: {va:.2f}% | LR: {lr:.2e}')
        if math.isnan(tl) or math.isnan(vl):
            print('NaN detected, stopping'); break
        scheduler.step(vl)
        if va > best_acc:
            best_acc, best_ep = va, ep
            torch.save({
                'epoch':ep,'model_state_dict':model.state_dict(),
                'optimizer_state_dict':optimizer.state_dict(),
                'val_acc':va,'val_loss':vl,
            }, Path(checkpoint_dir)/'fruit_cnn.pth')
        if es(vl):
            print(f'Early stopping at epoch {ep}'); break
        with open(log_path,'a',newline='') as f:
            csv.writer(f).writerow([ep,tl,ta,vl,va,lr])
    history['best_epoch'], history['best_val_acc'] = best_ep, best_acc
    print(f'Best val acc: {best_acc:.2f}% (epoch {best_ep})')
    return history

In [ ]:
%%writefile src/training/train.py
import torch, torch.nn as nn, torch.optim as optim
from src.utils.config import Config, set_seed, get_device
from src.data.fruit_dataset import get_dataloaders
from src.models.cnn_model import FruitCNN
from src.training.trainer import train_model, validate

def main():
    config = Config()
    set_seed(config.random_seed)
    device = get_device()
    print(f'Device: {device} | Classes: {config.num_classes} | Batch: {config.batch_size} | Epochs: {config.num_epochs}')
    tl, vl, tel = get_dataloaders(config.data_processed_dir, config.batch_size, config.img_size, config.val_split_from_train, config.random_seed)
    print(f'Loaders: train={len(tl)} val={len(vl)} test={len(tel)}')
    model = FruitCNN(config.num_classes, config.dropout_rate).to(device)
    print(f'Params: {sum(p.numel() for p in model.parameters()):,}')
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=config.scheduler_factor, patience=config.scheduler_patience, verbose=True)
    print('-'*50)
    history = train_model(model, tl, vl, criterion, optimizer, scheduler, config,
                          config.num_epochs, config.models_dir, config.models_dir/'training_log.csv')
    print('='*50)
    test_loss, test_acc = validate(model, tel, criterion, device)
    print(f'Test Accuracy: {test_acc:.2f}%')
    print(f'Best Val Accuracy: {history["best_val_acc"]:.2f}% (epoch {history["best_epoch"]})')
    print(f'Model saved to: models/fruit_cnn.pth')

if __name__ == '__main__':
    main()

In [ ]:
# 7. 数据预处理（筛选 15 类 → 分层分割）
from src.data.make_dataset import main as prep_data
prep_data()

In [ ]:
# 8. 开始训练
from src.training.train import main as train_main
train_main()

In [ ]:
# 9. 保存模型到 Google Drive
import shutil
shutil.copy('models/fruit_cnn.pth', f'{drive_models}/fruit_cnn.pth')
shutil.copy('models/training_log.csv', f'{drive_models}/training_log.csv')
print(f'模型已备份: {drive_models}/')
print('也可从 Colab 左侧 Files 面板右键下载 models/fruit_cnn.pth')

## 操作说明

**首次准备**：kaggle.com/settings → Create Token → 下载 `kaggle.json`

**每次训练**：
1. 左侧 Files 面板上传 `kaggle.json`
2. Runtime → T4 GPU
3. 顺序执行所有 Cell
4. 训练完成后模型自动备份到 Google Drive `MyDrive/fruit-model/`
5. 下载 `fruit_cnn.pth` 到本地 `models/` 目录